# Training for Infill Model
Imports, params, loading dataset

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))  # make src/ importable

import torch
from pathlib import Path
import json
from src.dataset import load_cached_paths, download_dataset
from src.model import build_model, build_infill_model, load_model, DEFAULT_CONFIG, append_epoch_to_config
from src.trainer import train
from src.infill_trainer import train_infill, INFILL_WEIGHTS_FILE
from src.tokenizer import (
    VOCAB_SIZE
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
print(f"Vocabulary size: {VOCAB_SIZE} (includes infill tokens)")

Using device: cpu
Vocabulary size: 415 (includes infill tokens)


In [1]:
# Infill Training Configuration
INFILL_EPOCHS = 2000
INFILL_BATCH_SIZE = 8
INFILL_LR = 1e-3
INFILL_MAX_FILES = 10000
FIM_RATIO = 0.5  # 50% FIM examples, 50% standard examples

In [5]:
# Load dataset
paths = load_cached_paths()
if len(paths) < INFILL_MAX_FILES:
    print(f"Downloading up to {INFILL_MAX_FILES} scores...")
    download_dataset(log=print, progress=lambda i, n: None, max_files=INFILL_MAX_FILES)
    paths = load_cached_paths()
print(f"Using {len(paths)} score files for infill training.")

Fetching file tree from GitHub API...
Found 1462 score files.


Downloading: 100%|██████████| 1462/1462 [00:00<00:00, 28384.49it/s]

Dataset cached: 1462 files in storage/data/openscore/


<hr>

### Build model

In [ ]:
# Build infill model with expanded vocabulary
print("Building infill model...")
infill_model, cfg = build_infill_model()
infill_model = infill_model.to(DEVICE)
print(f"Infill model parameters: {infill_model.num_params():,}")
print(f"Vocab size: {infill_model.cfg['vocab_size']}")

infill_log = []
def infill_log_fn(msg):
    print(msg)
    infill_log.append(msg)
        
def on_infill_epoch(epoch, train_loss, val_loss):
    print(f"  → epoch {epoch}  train={train_loss:.4f}  val={val_loss:.4f}")
    append_epoch_to_config(cfg, train_loss, val_loss)

Building infill model...
Infill model parameters: 3,892,224
Vocab size: 415


<hr>

### Train model

In [ ]:
stop_flag = [False]  # set stop_flag[0] = True to interrupt training
train_infill(
    model=infill_model,
    data_paths=paths[:INFILL_MAX_FILES],
    epochs=INFILL_EPOCHS,
    batch_size=INFILL_BATCH_SIZE,
    lr=INFILL_LR,
    device=DEVICE,
    log=infill_log_fn,
    on_epoch=on_infill_epoch,
    stop_flag=stop_flag,
    fim_ratio=FIM_RATIO,
)
print(f"\nInfill model saved to: {INFILL_WEIGHTS_FILE}")